# neuropt vs Optuna — Kaggle Spaceship Titanic

<a target="_blank" href="https://colab.research.google.com/github/loevlie/neuropt/blob/main/examples/neuropt_kaggle_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

An LLM reads your training curves and tunes your model. Here we compare **neuropt** vs **Optuna** on a real Kaggle competition using XGBoost — same eval budget, same search space.

No GPU needed. Runs in ~5 minutes.

In [ ]:
!pip install -q "neuropt[llm]" xgboost optuna kaggle

import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✓ API key loaded from Colab Secrets")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

## Load & prep data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import LabelEncoder

# Download data (requires Kaggle credentials — upload kaggle.json to Colab secrets or ~/.kaggle/)
try:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    from google.colab import userdata
    kaggle_json = userdata.get("KAGGLE_JSON")
    if kaggle_json:
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
            f.write(kaggle_json)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
except Exception:
    pass

if not os.path.exists("spaceship-titanic"):
    os.system("kaggle competitions download -c spaceship-titanic --force -q")
    os.system("unzip -qo spaceship-titanic.zip -d spaceship-titanic")

train = pd.read_csv("spaceship-titanic/train.csv")
test = pd.read_csv("spaceship-titanic/test.csv")
print(f"Train: {train.shape}, Test: {test.shape}")
train.head()

In [ ]:
def preprocess(df):
    """Basic feature engineering for Spaceship Titanic."""
    df = df.copy()

    # Extract cabin parts
    df[["cabin_deck", "cabin_num", "cabin_side"]] = df["Cabin"].str.split("/", expand=True)
    df["cabin_num"] = pd.to_numeric(df["cabin_num"], errors="coerce")

    # Group from PassengerId
    df["group"] = df["PassengerId"].str.split("_").str[0].astype(int)
    df["group_size"] = df.groupby("group")["group"].transform("count")

    # Spending features
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    df[spend_cols] = df[spend_cols].fillna(0)
    df["total_spend"] = df[spend_cols].sum(axis=1)
    df["no_spend"] = (df["total_spend"] == 0).astype(int)

    # Encode categoricals
    cat_cols = ["HomePlanet", "Destination", "cabin_deck", "cabin_side"]
    for col in cat_cols:
        df[col] = df[col].fillna("Unknown")
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

    df["CryoSleep"] = df["CryoSleep"].map({True: 1, False: 0, "True": 1, "False": 0}).fillna(-1).astype(int)
    df["VIP"] = df["VIP"].map({True: 1, False: 0, "True": 1, "False": 0}).fillna(-1).astype(int)
    df["Age"] = df["Age"].fillna(df["Age"].median())

    feature_cols = ["HomePlanet", "CryoSleep", "cabin_deck", "cabin_num", "cabin_side",
                    "Destination", "Age", "VIP", "group_size",
                    "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck",
                    "total_spend", "no_spend"]
    return df[feature_cols].fillna(0)

X = preprocess(train)
y = train["Transported"].astype(int)
X_test = preprocess(test)

print(f"Features: {X.shape[1]}")
print(f"Target: {y.value_counts().to_dict()}")

## Define train function

Both neuropt and Optuna will call this same function. 5-fold CV, returns per-fold accuracy curves so the LLM can reason about variance.

In [ ]:
import xgboost as xgb

N_FOLDS = 5
SEED = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def train_fn(config):
    """Train XGBoost with given config, return 5-fold CV results."""
    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "verbosity": 0,
        "random_state": SEED,
    }
    # Map config to XGBoost params
    for k in ["max_depth", "n_estimators", "learning_rate", "subsample",
              "colsample_bytree", "reg_alpha", "reg_lambda", "min_child_weight",
              "gamma"]:
        if k in config:
            params[k] = config[k]

    fold_accs = []
    fold_losses = []
    for tr_idx, val_idx in skf.split(X, y):
        model = xgb.XGBClassifier(**params)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
                  eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
                  verbose=False)
        preds = model.predict(X.iloc[val_idx])
        probs = model.predict_proba(X.iloc[val_idx])[:, 1]
        fold_accs.append(accuracy_score(y.iloc[val_idx], preds))
        fold_losses.append(log_loss(y.iloc[val_idx], probs))

    return {
        "score": np.mean(fold_losses),      # neuropt minimizes this
        "accuracy": np.mean(fold_accs),
        "train_losses": fold_losses,         # per-fold losses → the LLM sees variance
        "val_losses": fold_losses,
        "val_accuracies": fold_accs,
    }

# Quick sanity check with defaults
result = train_fn({"max_depth": 6, "n_estimators": 100, "learning_rate": 0.3})
print(f"Default XGBoost: acc={result['accuracy']:.4f}, log_loss={result['score']:.4f}")

## Run neuropt (15 evals)

In [ ]:
from neuropt import ArchSearch

search_space = {
    "max_depth":        (3, 10),
    "n_estimators":     (50, 500),
    "learning_rate":    (0.01, 0.3),
    "subsample":        (0.5, 1.0),
    "colsample_bytree": (0.5, 1.0),
    "reg_alpha":        (1e-5, 10.0),
    "reg_lambda":       (1e-5, 10.0),
    "min_child_weight": (1, 10),
    "gamma":            (1e-5, 5.0),
}

N_EVALS = 15

search = ArchSearch(
    train_fn=train_fn,
    search_space=search_space,
    backend="claude",
    log_path="neuropt_search.jsonl",
)
search.run(max_evals=N_EVALS, resume=False)

neuropt_best_score = search.best_score
neuropt_best_acc = search.best_accuracy
neuropt_best_config = search.best_config
print(f"\nneuropt best: log_loss={neuropt_best_score:.4f}, acc={neuropt_best_acc:.4f}")

## Run Optuna (same 15 evals, same search space)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def optuna_objective(trial):
    config = {
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "n_estimators":     trial.suggest_int("n_estimators", 50, 500),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-5, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma":            trial.suggest_float("gamma", 1e-5, 5.0, log=True),
    }
    result = train_fn(config)
    trial.set_user_attr("accuracy", result["accuracy"])
    return result["score"]

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(
    seed=SEED, n_startup_trials=3))
study.optimize(optuna_objective, n_trials=N_EVALS, show_progress_bar=True)

optuna_best_score = study.best_value
optuna_best_acc = study.best_trial.user_attrs["accuracy"]
print(f"\nOptuna best: log_loss={optuna_best_score:.4f}, acc={optuna_best_acc:.4f}")

## Compare results

In [ ]:
import json
import matplotlib.pyplot as plt

# Load neuropt history
with open("neuropt_search.jsonl") as f:
    neuropt_history = [json.loads(line) for line in f]
neuropt_scores = [r["score"] for r in neuropt_history]

# Load optuna history
optuna_scores = [t.value for t in study.trials]

# Best-so-far curves
def best_so_far(scores):
    best, out = float("inf"), []
    for s in scores:
        if s < best: best = s
        out.append(best)
    return out

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence
ax = axes[0]
evals = np.arange(1, N_EVALS + 1)
ax.plot(evals, best_so_far(neuropt_scores), "o-", color="#d32f2f", linewidth=2.5, markersize=6, label="neuropt (Claude)")
ax.plot(evals, best_so_far(optuna_scores), "s-", color="#757575", linewidth=2.5, markersize=6, label="Optuna TPE")
ax.set_xlabel("Evaluation #", fontsize=12)
ax.set_ylabel("Best CV Log Loss", fontsize=12)
ax.set_title("Convergence — Spaceship Titanic (XGBoost)", fontweight="bold", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Summary table
ax = axes[1]
ax.axis("off")
table_data = [
    ["", "neuropt", "Optuna"],
    ["Best Log Loss", f"{neuropt_best_score:.4f}", f"{optuna_best_score:.4f}"],
    ["Best Accuracy", f"{neuropt_best_acc:.4f}", f"{optuna_best_acc:.4f}"],
    ["Evals", str(N_EVALS), str(N_EVALS)],
]
table = ax.table(cellText=table_data, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(13)
table.scale(1, 2)
# Bold header row
for j in range(3):
    table[0, j].set_text_props(fontweight="bold")
# Highlight winner
winner_col = 1 if neuropt_best_score < optuna_best_score else 2
for i in range(1, 4):
    table[i, winner_col].set_facecolor("#c8e6c9")
ax.set_title("Head-to-Head", fontweight="bold", fontsize=13)

plt.tight_layout()
plt.show()

print(f"\nneuropt API cost: shown in the summary above (typically $0.01–0.05)")
print(f"Optuna API cost: $0.00 (no LLM needed)")

## Generate Kaggle submission with the best config

In [ ]:
# Train final model on all training data with the best config
best = neuropt_best_config
params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "verbosity": 0,
    "random_state": SEED,
}
for k in ["max_depth", "n_estimators", "learning_rate", "subsample",
          "colsample_bytree", "reg_alpha", "reg_lambda", "min_child_weight", "gamma"]:
    if k in best:
        params[k] = best[k]

final_model = xgb.XGBClassifier(**params)
final_model.fit(X, y)

preds = final_model.predict(X_test)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": preds.astype(bool)
})
submission.to_csv("submission.csv", index=False)
print(f"Submission saved: {len(submission)} rows")
print(submission.head())